# Catch-Up Sleep and Depressive Symptoms: Data Cleaning
**Eric Wang and Mary Yang | INFO 2950**

This separate notebook produces the analysis-ready data for Phase III. It preserves the cleaning and eligibility rules in `Phase2_Eric_Wang_Mary_Yang_final.ipynb`; only the organization and output location have changed.

**Run order:** Download the complete repository. Open this notebook from the repository root (the folder containing `data/`) and use Restart Kernel and Run All. Required packages: numpy, pandas, and DuckDB. Run this notebook before `Phase3_Eric_Wang_Mary_Yang.ipynb`.

The original XPT files and the Phase II processed files are not modified. Phase III outputs are saved separately under `data/processed/phase3/`.

## Provenance
Source: `Phase2_Eric_Wang_Mary_Yang_final.ipynb` (code cell 4, zero-based index).

Source SHA-256: `3e4c3e8489c132cabb4ef8461b726b3edd37efd49913b97a6fe0ba39c7fca3b8`.

Extracted on November 10, 2026. The two preregistered analyses are preserved separately in the Phase III main notebook. No statistical model is fit during cleaning.

## Data Collection and Cleaning

### Data Collection
We downloaded three public NHANES 2017–2018 datasets from the CDC website on October 15, 2026:

- Sleep Disorders (SLQ_J): usual sleep duration on weekdays/workdays (SLD012) and weekends/non-workdays (SLD013).
- Depression Screener (DPQ_J): nine depressive symptom items (DPQ010–DPQ090).
- Demographics (DEMO_J): participant age, sex, survey weights, and survey design variables.

The original SAS transport (.xpt) files are stored in data/raw and are preserved without modification. We use SEQN, the participant identifier, to merge the datasets.

Documentation:
- Sleep: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/SLQ_J.htm
- Depression: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DPQ_J.htm
- Demographics: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm

## 1. Set Up
Create the separate output directory if necessary. Relative paths assume the repository root is the working directory; do not launch this notebook from an unrelated folder.

In [1]:
import os
import numpy as np
import pandas as pd
import duckdb

if not os.path.isfile("data/raw/SLQ_J.xpt"):
    raise FileNotFoundError(
        "Open this notebook from the repository root containing data/raw/. "
        "Download the complete repository, not only the notebook."
    )
os.makedirs("data/processed/phase3", exist_ok=True)

## 2. Read and Merge
Read the three original XPT files and check that SEQN is unique and nonmissing. Left joins retain every sleep-questionnaire participant while attaching matching depression and demographic records. Save the merged intermediate dataset and record the starting sample size.

In [2]:
# Read the three original NHANES datasets.
sleep = pd.read_sas("data/raw/SLQ_J.xpt", format="xport")
depression = pd.read_sas("data/raw/DPQ_J.xpt", format="xport")
demographics = pd.read_sas("data/raw/DEMO_J.xpt", format="xport")

# Check that participant IDs are nonmissing and unique before merging.
for table in [sleep, depression, demographics]:
    assert table["SEQN"].notna().all(), "Missing SEQN"
    assert table["SEQN"].is_unique, "Duplicate SEQN"

# Keep all sleep participants and attach matching questionnaire records.
df = duckdb.sql("""
    SELECT
        s.SEQN, s.SLD012, s.SLD013,
        m.RIDAGEYR, m.RIAGENDR,
        m.WTMEC2YR, m.SDMVPSU, m.SDMVSTRA,
        d.DPQ010, d.DPQ020, d.DPQ030,
        d.DPQ040, d.DPQ050, d.DPQ060,
        d.DPQ070, d.DPQ080, d.DPQ090
    FROM sleep AS s
    LEFT JOIN depression AS d ON s.SEQN = d.SEQN
    LEFT JOIN demographics AS m ON s.SEQN = m.SEQN
""").df()
# Confirm that merging did not duplicate or remove sleep participants.
assert len(df) == len(sleep), "Unexpected row count after merging"

# Save the merged dataset and record the starting sample size.
df.to_csv("data/processed/phase3/01_merged.csv", index=False)
counts = {"Merged sleep sample": len(df)}

## 3. Recode Depression Items and Calculate the Score
Preserve the Phase II correction that restores near-zero numerical representations of zero in the nine discrete symptom items. Keep only scores 0, 1, 2, or 3; other responses, including refused (7) and do not know (9), become missing. SQL addition produces a missing total if any of the nine items is missing. Save the recoded intermediate dataset.

In [3]:
items = [
    "DPQ010", "DPQ020", "DPQ030",
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]
# Restore zeros read as tiny numerical artifacts in these discrete items.
df[items] = df[items].mask(df[items].abs() < 1e-12, 0)
# Keep valid responses (0-3); convert all other codes to missing.
df[items] = df[items].where(df[items].isin([0, 1, 2, 3]))

# Calculate the total; SQL addition returns NULL if any item is missing.
df = duckdb.sql("""
    SELECT *,
        DPQ010 + DPQ020 + DPQ030 + DPQ040 + DPQ050
        + DPQ060 + DPQ070 + DPQ080 + DPQ090 AS depression_score
    FROM df
""").df()
df.to_csv("data/processed/phase3/02_recoded.csv", index=False)

## 4. Apply Eligibility Rules
Apply the original rules sequentially: age at least 18; usable sleep durations in both variables; fewer than seven hours of workday sleep; and complete responses for the depression total. The range 3-13.5 excludes missing values and the bottom- and top-coded sleep responses 2 and 14. Record the remaining sample size after every rule.

In [4]:
# Define eligibility rules; the sleep range excludes missing values and codes 2/14.
filters = [
    ("Adults aged 18+", "RIDAGEYR >= 18"),
    ("Exact sleep durations available",
     "SLD012 BETWEEN 3 AND 13.5 AND SLD013 BETWEEN 3 AND 13.5"),
    ("Workday sleep below 7 hours", "SLD012 < 7"),
    ("Complete depression score", "depression_score IS NOT NULL")
]
# Apply filters sequentially and record the remaining sample size.
for label, condition in filters:
    df = duckdb.sql(f"SELECT * FROM df WHERE {condition}").df()
    counts[label] = len(df)

## 5. Calculate Catch-Up Sleep
Subtract workday sleep from non-workday sleep. Keep negative differences because they represent participants who report less sleep on non-workdays; do not remove them as outliers. Keep weights and design variables for provenance and possible explicitly labeled later analyses. Their presence does not make the planned unweighted analyses survey-weighted.

In [5]:
# Calculate non-workday minus workday sleep, retaining negative differences.
df = duckdb.sql("""
    SELECT *, SLD013 - SLD012 AS catch_up_sleep
    FROM df
""").df()

## 6. Save the Analysis-Ready Data
Export the cleaned dataset and sample-selection counts. The Phase III main notebook reads these files directly and does not repeat the cleaning procedure.

Outputs: `01_merged.csv`, `02_recoded.csv`, `03_analysis_ready.csv`, and `sample_counts.csv`, all in `data/processed/phase3/`. The Phase II reference sample contained 1,252 participants and 19 columns; any unexpected discrepancy should be investigated before analysis.

In [6]:
# Save the analysis-ready dataset and sample-selection counts.
df.to_csv("data/processed/phase3/03_analysis_ready.csv", index=False)
sample_counts = pd.Series(counts, name="Participants").to_frame()
sample_counts.to_csv("data/processed/phase3/sample_counts.csv")
print(sample_counts)
print(df.head())

                                 Participants
Merged sleep sample                      6161
Adults aged 18+                          5856
Exact sleep durations available          5726
Workday sleep below 7 hours              1402
Complete depression score                1252
      SEQN  SLD012  SLD013  RIDAGEYR  RIAGENDR       WTMEC2YR  SDMVPSU  \
0  93713.0     5.5     7.0      67.0       1.0  166841.660681      1.0   
1  93715.0     5.0     5.0      71.0       1.0    9250.904672      2.0   
2  93722.0     6.0     7.0      60.0       2.0  126309.823276      1.0   
3  93723.0     6.0     7.0      64.0       1.0  229129.236930      1.0   
4  93728.0     6.5     8.0      53.0       1.0   31532.904509      1.0   

   SDMVSTRA  DPQ010  DPQ020  DPQ030  DPQ040  DPQ050  DPQ060  DPQ070  DPQ080  \
0     140.0     1.0     1.0     2.0     0.0     1.0     1.0     2.0     0.0   
1     139.0     0.0     0.0     3.0     1.0     0.0     0.0     0.0     0.0   
2     139.0     0.0     0.0     1.0     0.